# NextMove — Tennis Detector: Transfer Learning on Kaggle GPU
Fine-tunes a COCO-pretrained YOLO11 backbone on a public tennis detection
dataset, then exports a Core ML model for the iOS app.

**Pipeline:** COCO-pretrained weights → fix labels → fine-tune → validate →
export to Core ML (`TennisDetector`).

This model powers **Tennis** and, via a transfer stand-in, **Badminton**
(fast small projectile, net-divided court). Padel has its own dedicated model.

> Enable GPU: Settings → Accelerator → **GPU T4 x2**. Turn **Internet** On.
>
> **Recommended:** run via **Save Version → Save & Run All (Commit)** so the
> session can't idle-timeout mid-training and the artifacts zip is waiting in
> the output when it finishes.
>
> The `tennis ball` class is small and fast — the hardest class. This notebook
> trains at higher resolution with small-object augmentation to help it. Judge
> success by the per-class **ball mAP50**, not just the overall number.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow coremltools
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Download the tennis dataset
Get a free Roboflow API key: https://app.roboflow.com/settings/api

Set `WORKSPACE / PROJECT / VERSION` in the cell below to your chosen Roboflow
tennis dataset. You want an **object detection** set with `ball` + `player`
(or `person`) classes. The cell prints the class list so you can confirm.

The download is wrapped in a retry loop — Roboflow's export can stall at 50%
on the first request while it generates the export; a retry returns the cached
result.

In [ ]:
import time
from roboflow import Roboflow

ROBOFLOW_API_KEY = ''  # <-- paste your free key

# Dataset: https://universe.roboflow.com/hyowon-p233u/tennis-vuufj
WORKSPACE = 'hyowon-p233u'
PROJECT   = 'tennis-vuufj'
VERSION   = None   # None = auto-resolve latest; or pin a number (e.g. 1)

proj = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(WORKSPACE).project(PROJECT)
if VERSION is None:
    VERSION = int(str(proj.versions()[0].version).split('/')[-1])
    print('Resolved latest version:', VERSION)
version = proj.version(VERSION)

dataset = None
for attempt in range(3):
    try:
        dataset = version.download('yolov8', overwrite=True)
        break
    except Exception as e:
        print(f'attempt {attempt+1} failed: {e}; retrying in 15s')
        time.sleep(15)
assert dataset is not None, 'dataset download failed after retries'
print('Dataset at:', dataset.location)

# Confirm the classes are what we expect (want ball + player/person).
import yaml
with open(f'{dataset.location}/data.yaml') as fh:
    cfg = yaml.safe_load(fh)
print('classes:', cfg.get('names'), '| nc:', cfg.get('nc'))

## 2b. Fix the labels (safety net)
Some Roboflow exports mix **segmentation polygons** with detection boxes in the
same label files; YOLO's detection trainer then silently drops those images.
This converts any polygon rows to tight bounding boxes. If the export is
already pure detection, it's a harmless no-op (0 polygons converted).

In [ ]:
from pathlib import Path

def poly_to_bbox(coords):
    xs, ys = coords[0::2], coords[1::2]
    return (min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)
clamp01 = lambda v: min(max(v, 0.0), 1.0)

root = Path(dataset.location)
converted = rows = files = 0
for split in ('train', 'valid', 'test'):
    ldir = root/split/'labels'
    if not ldir.is_dir():
        continue
    for f in ldir.glob('*.txt'):
        out = []
        for line in f.read_text().splitlines():
            p = line.split()
            if not p:
                continue
            cls, nums = p[0], [float(x) for x in p[1:]]
            if len(nums) == 4:
                xc, yc, w, h = nums
            elif len(nums) >= 6 and len(nums) % 2 == 0:
                xc, yc, w, h = poly_to_bbox(nums); converted += 1
            else:
                continue
            xc, yc, w, h = map(clamp01, (xc, yc, w, h))
            if w > 0 and h > 0:
                out.append(f'{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')
        f.write_text('\n'.join(out) + ('\n' if out else ''))
        files += 1; rows += len(out)
print(f'Checked {files} label files, {rows} boxes ({converted} polygons converted).')

## 3. Train (ball-focused)
Higher resolution (960) + small-object augmentation (copy-paste, mixup) +
heavier box-loss weight to help the tiny ball. `yolo11s` has more capacity than
nano and is still mobile-friendly. `save_period=10` checkpoints every 10 epochs
so a restart leaves recoverable weights.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=60, imgsz=960, batch=8, patience=25,
    optimizer='AdamW', lr0=0.001, lrf=0.01,
    mosaic=1.0, close_mosaic=10, scale=0.5, translate=0.1,
    copy_paste=0.3, mixup=0.1, fliplr=0.5, flipud=0.0, box=9.0,
    device=0, project='runs', name='tennis_detector', exist_ok=True,
    save_period=10,
)
print('training done')

## 4. Validate — honest per-class numbers
Locates `best.pt` wherever Ultralytics saved it (the exact path varies by
version, which is why we glob instead of hardcoding).

In [ ]:
import glob
from ultralytics import YOLO

cands = glob.glob('/kaggle/working/**/tennis_detector/weights/best.pt', recursive=True)
assert cands, 'best.pt not found — did training finish?'
best_path = cands[0]
run_dir = Path(best_path).parents[1]   # .../tennis_detector
print('Weights:', best_path)

best = YOLO(best_path)
m = best.val(data=f'{dataset.location}/data.yaml', imgsz=960, plots=True)
names = best.names
ap = {int(i): a for i, a in zip(m.box.ap_class_index, m.box.ap50)}
print(f'\nOverall mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}  P={m.box.mp:.3f}  R={m.box.mr:.3f}')
for i in sorted(ap):
    print(f'  {names[i]:14s} mAP50={ap[i]:.3f}')

## 5. Export to Core ML
Export imgsz **must match** the training imgsz (960). `nms=True` bakes in
non-max suppression so the Swift side stays simple.

In [ ]:
best.export(format='coreml', nms=True, imgsz=960)
print('Core ML exported next to best.pt as best.mlpackage')

## 6. Package + download
Bundles weights + Core ML model (renamed `TennisDetector_v1.mlpackage`) + the
full training run (curves, confusion matrix, results.csv) into one zip.

In [ ]:
import shutil

out = Path('/kaggle/working/tennis_artifacts')
shutil.rmtree(out, ignore_errors=True); out.mkdir(parents=True)

shutil.copy2(best_path, out/'best.pt')
print('added best.pt')

mlpkg = run_dir/'weights/best.mlpackage'
if mlpkg.exists():
    shutil.copytree(mlpkg, out/'TennisDetector_v1.mlpackage')
    print('added TennisDetector_v1.mlpackage')
else:
    print('WARNING: best.mlpackage not found — run the export cell first.')

shutil.copytree(run_dir, out/'training_run', ignore=shutil.ignore_patterns('*.mlpackage'))
print('added training_run/')

z = shutil.make_archive('/kaggle/working/tennis_artifacts', 'zip', out)
print(f'\n✅ {z} ({Path(z).stat().st_size/1e6:.1f} MB)')

# ---- Download ----
# Interactive run: click this link to save the zip to your computer.
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/tennis_artifacts.zip'))
# Committed run (Save & Run All): the zip is a notebook OUTPUT. Pull it to your
# Mac with the Kaggle API (no browser clicking):
#   pip install kaggle          # once; put kaggle.json in ~/.kaggle/
#   kaggle kernels output <username>/<notebook-slug> -p ~/Downloads
print('\nThen: drop TennisDetector_v1.mlpackage into Models/Tennis/,')
print('add to Xcode Copy Bundle Resources. Powers Tennis + Badminton.')